# P2 Project Reimplementation — Natural Gradient Boosting

This notebook is the reproducible entry point for the required P2 paper reimplementation. The historical PyPI `ngboost==0.5.11` experiment is retained only as an **official-library baseline**. This notebook reconstructs the verified project P2 source, applies documented integration-only patches, runs mathematical/API tests, and evaluates Tasks C/R/L plus the multivariate paper experiment.


## 1. Locate the repository
The resolver checks the normal Colab checkout and the documented Google Drive locations.


In [ ]:
from pathlib import Path
import os, sys, subprocess
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass
candidates = [
    Path('/content/ML-Project-Football-Forecasting'),
    Path('/content/drive/MyDrive/ml_project'),
    Path('/content/drive/MyDrive/ML-Project-Football-Forecasting'),
    Path.cwd(), Path.cwd().parent,
]
required = Path('code/data_pipeline/results/Premier_League_2015_16_DATA_ONLY_FULL_REAL/data/gold/gold_prematch.parquet')
PROJECT_ROOT = next((p for p in candidates if (p / required).exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Repository not found. Set PROJECT_ROOT to the repository directory.')
PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT =', PROJECT_ROOT)


## 2. Install the pinned environment
The P2 environment intentionally does not install the external `ngboost` package. Dependencies imported by the supplied source, including SymPy, tqdm and lifelines, are pinned explicitly.


In [ ]:
req = PROJECT_ROOT / 'code/modeling/requirements_colab_p2_reimplementation.txt'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
print('Environment installed from', req)


## 3. Verify and prepare the P2 source
The preparation step verifies the committed source archive checksum and every pre-patch Python file against the SHA-256 manifest. A mismatch is an error; verification is never bypassed.


In [ ]:
modeling = PROJECT_ROOT / 'code/modeling'
subprocess.check_call([sys.executable, str(modeling / 'prepare_p2_reimplementation.py')])


## 4. Mathematical and compatibility tests
These tests include sklearn cloning/determinism, probability normalization, Normal finite-difference gradient verification, Fisher symmetry/positive-definiteness, direct natural-gradient solve agreement, multivariate positive-definite covariance, serialization, and the natural-vs-Euclidean gradient path.


In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(modeling)
subprocess.check_call([sys.executable, '-m', 'pytest', '-q', str(modeling / 'test_p2_reimplementation.py')], env=env)


## 5. Full P2 evaluation
The run preserves the chronological 251/77/46 match split. The 128 pre-match and 189 live MD1 feature contracts are reconstructed from the data-pipeline audit; the 18 P1 fields are taken from the frozen P1 table schema. The historical NGBoost model output is not used to define features or tune P2.


In [ ]:
runner = modeling / 'run_p2_reimplementation_final.py'
subprocess.check_call([sys.executable, str(runner), '--mode', 'full', '--project-root', str(PROJECT_ROOT)], env=env)


## 6. Review generated evidence
The output includes Task C/R/L metrics, row-level predictions, worst cases, reliability/calibration, frozen P2 live baselines, minute/phase curves, uncertainty coverage, multivariate diagnostics, true sampled peak RSS, latency percentiles, and a comparison against the separately labelled official-library baseline.


In [ ]:
import pandas as pd
from IPython.display import Markdown, display
out = PROJECT_ROOT / 'code/modeling/outputs/p2_reimplementation_full'
for name in ['p2_task_c_metrics.csv','p2_task_r_metrics.csv','p2_task_l_outcome_metrics.csv','p2_task_l_margin_metrics.csv','p2_multivariate_metrics.csv','p2_compute_and_peak_memory.csv','p2_vs_official_library.csv']:
    path = out / 'tables' / name
    if path.exists():
        print('\n', name); display(pd.read_csv(path))
display(Markdown((out / 'P2_REPORT_INSERT.md').read_text(encoding='utf-8')))
